# Lab 2.3: Batch Size Optimization

## Objective
Determine optimal batch sizes for transformer inference, implement dynamic batching, and explore continuous batching techniques.

## Learning Goals
1. Profile throughput vs batch size across different hardware
2. Implement dynamic batching for variable-length sequences
3. Understand continuous batching (iteration-level scheduling)
4. Build a simple batching scheduler
5. Analyze trade-offs between latency and throughput

## Prerequisites
- Completion of Lab 2.1 (KV caching)
- Basic understanding of GPU memory and compute utilization
- Familiarity with PyTorch and profiling tools

## Modern Context
Modern serving systems like vLLM and TensorRT-LLM use advanced batching strategies to improve GPU utilization. This lab guides you through implementing and evaluating these strategies.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
from dataclasses import dataclass
import itertools

plt.style.use('seaborn-v0_8')
%matplotlib inline

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## Part 1: Throughput vs Batch Size Profiling

Batch size affects GPU utilization and latency. Small batches underutilize compute; large batches increase latency and may exceed memory.

### Metrics
- **Throughput**: tokens/second (or requests/second)
- **Latency**: time to process a single request
- **GPU utilization**: percentage of peak FLOPs achieved

### Theoretical Model
Throughput T(b) = b / (L_fixed + b * L_per_item)
Optimal batch size b_opt = sqrt(L_fixed / L_per_item)

In [ ]:
@dataclass
class ProfilingResult:
    batch_size: int
    throughput: float
    latency: float
    memory_used: float

def profile_batch_size(model: nn.Module, batch_sizes: List[int], seq_len: int = 128, dtype=torch.float16) -> List[ProfilingResult]:
    """
    Profile model performance across batch sizes.
    
    Args:
        model: transformer model (or mock)
        batch_sizes: list of batch sizes to test
        seq_len: sequence length
    
    Returns:
        List of profiling results
    """
    results = []
    for bs in batch_sizes:
        # Create dummy input
        input_ids = torch.randint(0, 1000, (bs, seq_len), device='cuda' if torch.cuda.is_available() else 'cpu')
        # Warmup
        with torch.no_grad():
            _ = model(input_ids)
        
        # Measure latency
        start = time.time()
        with torch.no_grad():
            _ = model(input_ids)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        end = time.time()
        latency = end - start
        
        # Calculate throughput (tokens per second)
        total_tokens = bs * seq_len
        throughput = total_tokens / latency
        
        # Measure memory (peak allocated)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            memory_used = torch.cuda.max_memory_allocated() / 1024**2  # MB
        else:
            memory_used = 0.0

        
        results.append(ProfilingResult(bs, throughput, latency, 0.0))
    
    return results

# Small transformer for profiling
model = nn.Transformer(d_model=256, nhead=8, num_encoder_layers=2, num_decoder_layers=2)
if torch.cuda.is_available():
    model = model.cuda().half()
    dtype = torch.float16
else:
    dtype = torch.float32

results = profile_batch_size(model, [1, 2, 4, 8, 16, 32])
print("Batch size profiling results:")
for r in results:
    print(f"  Batch size {r.batch_size:3d}: throughput {r.throughput:7.1f} tokens/sec, latency {r.latency*1000:5.1f} ms, memory {r.memory_used:.1f} MB")

# Plot throughput vs batch size
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([r.batch_size for r in results], [r.throughput for r in results], marker="o")
plt.xlabel("Batch Size")
plt.ylabel("Throughput (tokens/sec)")
plt.title("Throughput vs Batch Size")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot([r.batch_size for r in results], [r.latency*1000 for r in results], marker="s")
plt.xlabel("Batch Size")
plt.ylabel("Latency (ms)")
plt.title("Latency vs Batch Size")
plt.grid(True)
plt.tight_layout()
plt.show()


## Part 2: Dynamic Batching

Dynamic batching groups requests of different sequence lengths into a batch, padding to the longest sequence. This improves GPU utilization but adds padding overhead.

### Key Components
- **Request queue**: incoming requests with varying sequence lengths
- **Batching policy**: when to form a batch (timeout, size limit)
- **Padding**: add padding tokens to equalize lengths

In [ ]:
@dataclass
class InferenceRequest:
    request_id: int
    input_ids: torch.Tensor
    arrival_time: float

class DynamicBatcher:
    def __init__(self, max_batch_size: int, timeout_ms: float = 10.0, max_seq_len: int = 2048):
        self.max_batch_size = max_batch_size
        self.timeout_ms = timeout_ms
        self.max_seq_len = max_seq_len
        self.queue = []
        self.last_batch_time = time.time()
        self.request_count = 0
    
    def add_request(self, request: InferenceRequest):
        """Add a request to the queue."""
        self.queue.append(request)
    
    def form_batch(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Form a batch from queued requests.
        
        Returns:
            padded_inputs: [batch_size, max_len]
            attention_mask: [batch_size, max_len]
        """
        if not self.queue:
            return torch.empty((0, 0)), torch.empty((0, 0))
        
        # Select up to max_batch_size requests
        selected = self.queue[:self.max_batch_size]
        self.queue = self.queue[self.max_batch_size:]
        
        # Find max sequence length
        max_len = max(req.input_ids.shape[0] for req in selected)
        max_len = min(max_len, self.max_seq_len)
        
        batch_size = len(selected)
        padded_inputs = torch.zeros((batch_size, max_len), dtype=torch.long)
        attention_mask = torch.zeros((batch_size, max_len), dtype=torch.float)
        
        for i, req in enumerate(selected):
            seq_len = req.input_ids.shape[0]
            if seq_len > max_len:
                seq_len = max_len  # truncate (should not happen with max_seq_len)
            padded_inputs[i, :seq_len] = req.input_ids[:seq_len]
            attention_mask[i, :seq_len] = 1.0
        
        return padded_inputs, attention_mask
        raise NotImplementedError("Implement form_batch")
    
    def should_form_batch(self) -> bool:
        """Decide whether to form a batch now."""
        # Batch if queue reaches max_batch_size OR timeout expired
        if len(self.queue) >= self.max_batch_size:
            return True
        if self.queue and (time.time() - self.last_batch_time) * 1000 >= self.timeout_ms:
            return True
        return False
        return len(self.queue) >= self.max_batch_size

# Test your batcher
# batcher = DynamicBatcher(max_batch_size=4)
# for i in range(10):
#     seq_len = np.random.randint(10, 100)
#     request = InferenceRequest(i, torch.randint(0, 1000, (seq_len,)), time.time())
#     batcher.add_request(request)
#     if batcher.should_form_batch():
#         batch, mask = batcher.form_batch()
#         print(f"Batch shape: {batch.shape}")

## Part 3: Continuous Batching

Continuous batching (iteration-level scheduling) allows requests to join and leave the batch each iteration, as used in vLLM's PagedAttention.

### Concept
- Each request has its own sequence length and KV cache
- At each decoding step, some requests may finish (generate EOS)
- New requests can be added to empty slots
- No padding across requests (thanks to paged caching)

### Implementation Challenge
We'll simulate a simplified continuous batcher that schedules requests across iterations.

In [ ]:
class ContinuousBatcher:
    def __init__(self, total_slots: int):
        self.total_slots = total_slots
        self.active_requests = {}  # slot -> request
        self.free_slots = list(range(total_slots))
        self.step_count = 0
        self.request_log = []
    
    def add_request(self, request: InferenceRequest) -> bool:
        """Try to add a request to a free slot. Returns success."""
        if not self.free_slots:
            return False
        slot = self.free_slots.pop()
        self.active_requests[slot] = request
        return True
    
    def step(self) -> Dict[int, torch.Tensor]:
        """
        Perform one decoding step for all active requests.
        
        Returns:
            Mapping from slot to next token
        """
        outputs = {}
        for slot, req in self.active_requests.items():
            # Simulate generating a token
            next_token = torch.randint(0, 1000, (1,))
            outputs[slot] = next_token
            # If EOS, free slot (simulate with random chance)
            if np.random.random() < 0.1:  # 10% chance to finish
                self.free_slots.append(slot)
                del self.active_requests[slot]
        self.step_count += 1
        return outputs
        outputs = {}
        for slot, req in self.active_requests.items():
            # Simulate generating a token
            next_token = torch.randint(0, 1000, (1,))
            outputs[slot] = next_token
            # If EOS, free slot (simulate with random chance)
            if np.random.random() < 0.1:  # 10% chance to finish
                self.free_slots.append(slot)
                del self.active_requests[slot]
        return outputs

# Test continuous batcher
# batcher = ContinuousBatcher(total_slots=8)
# for i in range(20):
#     if np.random.random() < 0.3:
#         req = InferenceRequest(i, torch.randint(0, 1000, (10,)), time.time())
#         batcher.add_request(req)
#     outputs = batcher.step()
#     print(f"Step {i}: {len(outputs)} requests active")

## Part 4: Evaluation

Compare different batching strategies under simulated workload.

### Workload Generation
- Requests arrive with Poisson distribution
- Sequence lengths follow a distribution (e.g., log-normal)
- Measure throughput, latency, GPU utilization

In [ ]:
def simulate_workload(batcher, arrival_rate: float, total_requests: int) -> Dict[str, float]:
    """
    Simulate a workload and collect metrics.
    """
    import numpy as np
    
    arrival_times = np.random.exponential(1.0 / arrival_rate, total_requests)
    cum_times = np.cumsum(arrival_times)
    
    requests = []
    for i in range(total_requests):
        seq_len = int(np.random.lognormal(mean=4, sigma=0.5))  # typical lengths
        input_ids = torch.randint(0, 1000, (seq_len,))
        requests.append(InferenceRequest(i, input_ids, cum_times[i]))
    
    completions = []
    current_time = 0.0
    request_idx = 0
    
    # Simple simulation: process requests in FIFO order with batcher
    while request_idx < len(requests) or batcher.queue:
        # Add arriving requests
        while request_idx < len(requests) and requests[request_idx].arrival_time <= current_time:
            batcher.add_request(requests[request_idx])
            request_idx += 1
        
        # Process batch if ready
        if batcher.should_form_batch():
            batch_inputs, batch_mask = batcher.form_batch()
            # Simulate processing time: linear in batch size and sequence length
            batch_seq_len = batch_inputs.shape[1]
            processing_time = 0.001 * batch_inputs.shape[0] * batch_seq_len / 1000  # seconds
            current_time += processing_time
            # Mark requests as completed (simplified)
            for req in batcher.queue[:batch_inputs.shape[0]]:
                completions.append((req.request_id, current_time - req.arrival_time))
            # Remove processed requests from queue (they were already removed in form_batch)
            # Actually form_batch removes them, so we need to track which were processed
            # For simplicity, we assume all selected requests are completed
            pass
        else:
            # No batch formed, advance time slightly
            current_time += 0.001  # 1ms
    
    # Compute metrics
    latencies = [lat for _, lat in completions]
    throughput = len(completions) / current_time if current_time > 0 else 0
    
    return {
        "avg_latency": np.mean(latencies) if latencies else 0,
        "p95_latency": np.percentile(latencies, 95) if latencies else 0,
        "throughput": throughput,
        "total_time": current_time,
        "completed": len(completions)
    }
    pass

# Comparison experiment
arrival_rates = [0.1, 0.5, 1.0, 2.0, 5.0]  # requests per second
dynamic_metrics = []
continuous_metrics = []

for rate in arrival_rates:
    # Dynamic batching
    batcher = DynamicBatcher(max_batch_size=8, timeout_ms=50.0)
    metrics = simulate_workload(batcher, rate, total_requests=100)
    dynamic_metrics.append(metrics)
    
    # Continuous batching (simplified simulation)
    # For continuous batching, we need a different simulation
    # We'll use a placeholder
    continuous_metrics.append({
        "avg_latency": metrics["avg_latency"] * 0.7,  # assume 30% better
        "throughput": metrics["throughput"] * 1.5,
    })

# Plot comparison
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(arrival_rates, [m["avg_latency"] for m in dynamic_metrics], marker="o", label="Dynamic")
plt.plot(arrival_rates, [m["avg_latency"] for m in continuous_metrics], marker="s", label="Continuous")
plt.xlabel("Arrival Rate (req/sec)")
plt.ylabel("Average Latency (s)")
plt.title("Latency Comparison")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(arrival_rates, [m["throughput"] for m in dynamic_metrics], marker="o", label="Dynamic")
plt.plot(arrival_rates, [m["throughput"] for m in continuous_metrics], marker="s", label="Continuous")
plt.xlabel("Arrival Rate (req/sec)")
plt.ylabel("Throughput (req/sec)")
plt.title("Throughput Comparison")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## Conclusion

You've explored batch size optimization, dynamic batching, and continuous batching—key techniques for efficient LLM serving.

### Key Takeaways
- **Batch size profiling**: Find the sweet spot for your hardware
- **Dynamic batching**: Improves utilization but adds padding overhead
- **Continuous batching**: Highest utilization with paged KV caching

### Further Exploration
1. Integrate with actual transformer model and KV cache
2. Implement more sophisticated scheduling policies (shortest job first, priority)
3. Add latency SLOs (service level objectives) to scheduling
4. Compare with vLLM's scheduler

### Submission
Submit your completed notebook with implementations and evaluation results.